# InfernoTactics v8: Fire-Relative RL

This notebook runs the v8 generalization experiment. The policy uses semantic targets such as `active_fire`, `adjacent_fuel`, and `threatened_population` instead of memorizing absolute zone IDs. A new WUI ignition is sampled every episode while Mandeville Canyon and Getty View Park remain held out.

The Colab kernel executes this notebook, but it does not automatically expose the files on the local computer. Store the project in Google Drive or clone the repository, then set `PROJECT_ROOT` in the setup cell.

In [2]:
import glob
import os
import sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# Change this to the Drive folder containing src/, data/, and requirements.txt.
PROJECT_ROOT = '/content/drive/MyDrive/InfernoTactics/infernotactics'

# If the repository is already cloned under /content, this fallback can find it.
if not os.path.exists(os.path.join(PROJECT_ROOT, 'src', 'train', 'train_relative.py')):
    matches = glob.glob('/content/**/src/train/train_relative.py', recursive=True)
    if matches:
        PROJECT_ROOT = str(Path(matches[0]).parents[2])

PROJECT_ROOT = os.path.abspath(PROJECT_ROOT)
required = [
    os.path.join(PROJECT_ROOT, 'src', 'env', 'inferno_env.py'),
    os.path.join(PROJECT_ROOT, 'src', 'train', 'train_relative.py'),
    os.path.join(PROJECT_ROOT, 'data', 'grid_static.npy'),
]
if not all(os.path.exists(path) for path in required):
    raise RuntimeError('PROJECT_ROOT must point to the project folder containing src/ and data/.')

os.environ['PYTHONPATH'] = PROJECT_ROOT + os.pathsep + os.environ.get('PYTHONPATH', '')
sys.path.insert(0, PROJECT_ROOT)
print('PROJECT_ROOT:', PROJECT_ROOT)

Mounted at /content/drive


RuntimeError: PROJECT_ROOT must point to the project folder containing src/ and data/.

In [ ]:
# Install the project dependencies in the connected Colab kernel.
%pip install -q -r {os.path.join(PROJECT_ROOT, 'requirements.txt')}

## Verify Relative Actions

The same semantic action must resolve to different absolute zones when the ignition moves. This is the core invariant missing from the original policy.

In [ ]:
from src.env.inferno_env import InfernoEnv
from src.train.relative_actions import TARGET_TYPES, resolve_relative_targets

env = InfernoEnv(seed=8200)
obs_a = env.reset(ignition_point=(207, 222), seed=1)
zones_a, _ = resolve_relative_targets(env, obs_a)
obs_b = env.reset(ignition_point=(57, 371), seed=1)
zones_b, _ = resolve_relative_targets(env, obs_b)
active = TARGET_TYPES.index('active_fire')
print('active_fire zone at Skull Rock:', zones_a[0, active])
print('active_fire zone at Mandeville:', zones_b[0, active])
assert zones_a[0, active] != zones_b[0, active]
print('Relative-action invariant passed.')

## Smoke Test

Run this first. It checks model construction, randomized ignition sampling, action decoding, forward/backward passes, and held-out evaluation.

In [ ]:
import subprocess

run_env = os.environ.copy()
run_env.update({
    'INFERNO_N_EPISODES': '10',
    'INFERNO_V8_EVAL_EVERY': '3',
    'INFERNO_RUN_TAG': 'relative_v8_smoke',
    'PYTHONPATH': PROJECT_ROOT + os.pathsep + run_env.get('PYTHONPATH', ''),
})
subprocess.run([sys.executable, '-m', 'src.train.train_relative'], cwd=PROJECT_ROOT, env=run_env, check=True)

## Full Training

Run this after the smoke test succeeds. The default is 2,000 randomized-ignition episodes. Checkpoints are written under the project folder in `models/checkpoints_relative_v8_full`.

In [ ]:
N_EPISODES = 2000
RUN_TAG = 'relative_v8_full'

run_env = os.environ.copy()
run_env.update({
    'INFERNO_N_EPISODES': str(N_EPISODES),
    'INFERNO_V8_EVAL_EVERY': '50',
    'INFERNO_RUN_TAG': RUN_TAG,
    'PYTHONPATH': PROJECT_ROOT + os.pathsep + run_env.get('PYTHONPATH', ''),
})
subprocess.run([sys.executable, '-m', 'src.train.train_relative'], cwd=PROJECT_ROOT, env=run_env, check=True)

In [ ]:
from pathlib import Path
checkpoint_dir = Path(PROJECT_ROOT) / 'models' / 'checkpoints_relative_v8_full'
print('Checkpoint directory:', checkpoint_dir)
print('Checkpoints:', sorted(path.name for path in checkpoint_dir.glob('*.pt'))[-10:])